In [3]:
import sys
!{sys.executable} -m pip install -q conllu scikit-learn

import urllib.request
import conllu
import math
from collections import defaultdict, Counter
from sklearn.metrics import accuracy_score, classification_report

train_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu"
test_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu"

urllib.request.urlretrieve(train_url, "train.conllu")
urllib.request.urlretrieve(test_url, "test.conllu")

with open("train.conllu", "r", encoding="utf-8") as f:
    train_data = conllu.parse(f.read())

with open("test.conllu", "r", encoding="utf-8") as f:
    test_data = conllu.parse(f.read())

transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

for sentence in train_data:
    previous_tag = "<START>"

    for token in sentence:
        word = token["form"].lower()
        tag = token["upos"]

        transition_counts[previous_tag][tag] += 1
        emission_counts[tag][word] += 1
        tag_counts[tag] += 1

        previous_tag = tag

    transition_counts[previous_tag]["<END>"] += 1

tags = list(tag_counts.keys())

def transition_probability(previous_tag, tag):
    total = sum(transition_counts[previous_tag].values())
    return (transition_counts[previous_tag][tag] + 1) / (total + len(tags) + 1)

def emission_probability(tag, word):
    total = tag_counts[tag]
    return (emission_counts[tag][word] + 1) / (total + len(emission_counts[tag]) + 1)

def viterbi(words):
    words = [word.lower() for word in words]
    dp = [{}]
    backpointer = [{}]

    for tag in tags:
        dp[0][tag] = (
            math.log(transition_probability("<START>", tag))
            + math.log(emission_probability(tag, words[0]))
        )
        backpointer[0][tag] = None

    for i in range(1, len(words)):
        dp.append({})
        backpointer.append({})

        for current_tag in tags:
            best_score = float("-inf")
            best_previous = None

            for previous_tag in tags:
                score = (
                    dp[i-1][previous_tag]
                    + math.log(transition_probability(previous_tag, current_tag))
                    + math.log(emission_probability(current_tag, words[i]))
                )

                if score > best_score:
                    best_score = score
                    best_previous = previous_tag

            dp[i][current_tag] = best_score
            backpointer[i][current_tag] = best_previous

    best_last_tag = max(dp[-1], key=dp[-1].get)

    predicted_tags = [best_last_tag]

    for i in range(len(words) - 1, 0, -1):
        predicted_tags.append(backpointer[i][predicted_tags[-1]])

    predicted_tags.reverse()

    return predicted_tags

sentence = input("Enter a sentence: ")
words = sentence.split()

predicted = viterbi(words)

print("\nOutput:")

for word, tag in zip(words, predicted):
    print(f"{word} → {tag}")

predicted = viterbi(words)

print("Input:", sentence)
print("\nOutput:")

for word, tag in zip(words, predicted):
    print(f"{word} → {tag}")

y_true = []
y_pred = []

for sentence in test_data:
    words = [token["form"] for token in sentence]
    actual_tags = [token["upos"] for token in sentence]

    predicted_tags = viterbi(words)

    y_true.extend(actual_tags)
    y_pred.extend(predicted_tags)

accuracy = accuracy_score(y_true, y_pred)

print("\nPOS Tagging Accuracy on Test Dataset:")
print(f"{accuracy * 100:.2f}%")

print("\nEvaluation Report:")
print(classification_report(y_true, y_pred, zero_division=0))

Enter a sentence:  The cat is sleeping.



Output:
The → DET
cat → NOUN
is → AUX
sleeping. → PART
Input: The cat is sleeping.

Output:
The → DET
cat → NOUN
is → AUX
sleeping. → PART

POS Tagging Accuracy on Test Dataset:
66.50%

Evaluation Report:
              precision    recall  f1-score   support

         ADJ       0.88      0.57      0.69      1788
         ADP       0.91      0.74      0.81      2025
         ADV       0.83      0.68      0.75      1191
         AUX       0.89      0.87      0.88      1543
       CCONJ       0.95      0.71      0.81       736
         DET       0.94      0.87      0.90      1897
        INTJ       0.22      0.83      0.35       121
        NOUN       0.92      0.52      0.67      4123
         NUM       0.69      0.35      0.46       542
        PART       0.89      0.82      0.85       649
        PRON       0.93      0.90      0.91      2165
       PROPN       0.93      0.19      0.31      2075
       PUNCT       0.99      0.79      0.88      3096
       SCONJ       0.72      0.61    